In [0]:
%pip install cloudpickle mlflow  catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.7/96.7 MB 125.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 113.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 601.6/601.6 kB 8.3 MB/s eta 0:00:00
  Attempting uninstall: blinker
    Found existing installation: blinker 1.7.0
    Not uninstalling blinker at /usr/lib/python3/dist-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-b5a955df-8f00-42ab-bc5b-20c521d43e6e
    Can't uninstall 'blinker'. No files were found to uninstall.
  Attempting uninstall: mlflow-skinny
    Found existing installation: mlflow-skinny 3.8.1
    Not uninstalling mlflow-skinny at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemera

In [0]:
# ============================================
# Notebook: 09_Model_Serving_Setup
# Purpose: Register model in Unity Catalog and create Serving Endpoint
# ============================================

import mlflow
import cloudpickle
import pandas as pd
import joblib
from mlflow.models import infer_signature

# ------------------------------
# 1. Load trained pipeline and required feature list
# ------------------------------
VOLUME_PATH = "/Volumes/workspace/default/ml_artifacts/"
with open(f"{VOLUME_PATH}delivery_delay_pipeline_final.pkl", "rb") as f:
    pipeline = cloudpickle.load(f)

# Load the required feature names in the correct order (as used during training)
required_features = joblib.load(f"{VOLUME_PATH}required_features.pkl")
print(f"✅ Loaded {len(required_features)} features")

# ------------------------------
# 2. Create a dictionary with all features (using realistic values)
# ------------------------------
# The keys must exactly match the feature names
sample_data = {
    "order_datetime": "17-05-2025 19:30",
    "platform_name": "blinkit",
    "product_category_name": "grocery",
    "order_value_inr": 450.0,
    "delivery_time_min": 25,
    "service_rating": 4,
    "sla_delay": "No",
    "refund_flag": 0,
    "Segment": "Loyalist",
    "customer_delay_rate": 0.24,
    "customer_avg_rating": 4.2,
    "customer_order_count": 5,
    "customer_refund_rate": 0.0
}

# Reorder columns to match the training order
# Convert to DataFrame with the exact column order
sample_input = pd.DataFrame([sample_data])[required_features]

# ------------------------------
# 3. Generate output and infer signature
# ------------------------------
sample_output = pipeline.predict(sample_input)
signature = infer_signature(sample_input, sample_output)

# ------------------------------
# 4. Register model in Unity Catalog
# ------------------------------
mlflow.set_registry_uri("databricks-uc")
with mlflow.start_run():
    mlflow.sklearn.log_model(
        pipeline,
        artifact_path="model",
        signature=signature,
        registered_model_name="workspace.default.delivery_delay_model"
    )
print("✅ Model registered in Unity Catalog")

✅ Loaded 13 features


/home/spark-9dae5b6f-6e8c-4bbc-91ee-a8/.ipykernel/2021/command-5713992097181489-3077150703:9: UserWarning: Parsing dates in %d-%m-%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more detai

Uploading artifacts:   0%|          | 0/9 [00:00<?, ?it/s]

🔗 Created version '1' of model 'workspace.default.delivery_delay_model': https://dbc-7eee2e54-d86b.cloud.databricks.com/explore/data/models/workspace/default/delivery_delay_model/version/1?o=7474658855447080


✅ Model registered in Unity Catalog
